In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/datathon-2026/sample_submission.csv
/kaggle/input/competitions/datathon-2026/test_x.csv
/kaggle/input/competitions/datathon-2026/train.csv


In [2]:
import pandas as pd
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score,root_mean_squared_error
from sklearn.model_selection import cross_val_score, KFold
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import pipeline

# Veri Setini Yükle

In [3]:
train_dataset = pd.read_csv("/kaggle/input/competitions/datathon-2026/train.csv")

In [4]:
train_dataset.columns

Index(['student_id', 'application_year', 'age', 'graduation_year',
       'department', 'university_tier', 'cgpa', 'english_exam_score',
       'attendance_rate', 'failed_courses_count', 'target_role',
       'coding_score', 'problem_solving_score', 'data_structures_score',
       'sql_score', 'machine_learning_score', 'backend_score',
       'frontend_score', 'cloud_score', 'devops_score',
       'project_quality_score', 'real_client_project_count',
       'internship_count', 'internship_duration_months',
       'freelance_project_count', 'hackathon_count', 'hackathon_awards',
       'portfolio_score', 'github_repo_count', 'github_avg_stars',
       'open_source_contribution_count', 'linkedin_profile_score',
       'cv_quality_score', 'technical_interview_score', 'hr_interview_score',
       'communication_score', 'teamwork_score', 'leadership_score',
       'presentation_score', 'certification_count', 'bootcamp_count',
       'applications_sent', 'interviews_attended', 'hobby',
     

In [5]:
for column in train_dataset.columns:
    print(f"{column},{train_dataset[column].nunique()}")

student_id,10000
application_year,8
age,13
graduation_year,9
department,7
university_tier,4
cgpa,200
english_exam_score,5160
attendance_rate,3418
failed_courses_count,9
target_role,11
coding_score,4417
problem_solving_score,4438
data_structures_score,4487
sql_score,7468
machine_learning_score,6902
backend_score,5805
frontend_score,5463
cloud_score,5701
devops_score,5742
project_quality_score,5596
real_client_project_count,9
internship_count,6
internship_duration_months,25
freelance_project_count,8
hackathon_count,7
hackathon_awards,4
portfolio_score,5525
github_repo_count,58
github_avg_stars,1339
open_source_contribution_count,66
linkedin_profile_score,5299
cv_quality_score,5437
technical_interview_score,5486
hr_interview_score,5204
communication_score,5519
teamwork_score,5406
leadership_score,5477
presentation_score,5541
certification_count,10
bootcamp_count,9
applications_sent,239
interviews_attended,28
hobby,8
preferred_social_media_platform,6
career_success_score,4402
mentor_feedba

## Eksik Verileri Giderme

In [6]:
train_dataset["english_exam_score"] = train_dataset["english_exam_score"].fillna(train_dataset["english_exam_score"].mean())
train_dataset["open_source_contribution_count"] = train_dataset["open_source_contribution_count"].fillna(train_dataset["open_source_contribution_count"].mean())
train_dataset["internship_duration_months"] = train_dataset["internship_duration_months"].fillna(train_dataset["internship_duration_months"].mean())
train_dataset["hr_interview_score"] = train_dataset["hr_interview_score"].fillna(train_dataset["hr_interview_score"].mean())
train_dataset["linkedin_profile_score"] = train_dataset["linkedin_profile_score"].fillna(train_dataset["linkedin_profile_score"].mean())
train_dataset["portfolio_score"] = train_dataset["portfolio_score"].fillna(train_dataset["portfolio_score"].mean())
train_dataset["github_avg_stars"] = train_dataset["github_avg_stars"].fillna(train_dataset["github_avg_stars"].mean())

## TF-ID

In [7]:
turkce_stopwords = [
    'acaba', 'altmış', 'ama', 'ancak', 'arada', 'artık', 'asla', 'aslında', 'aslında', 
    'ayrıca', 'az', 'bana', 'bazen', 'bazı', 'bazıları', 'belki', 'ben', 'benden', 
    'beni', 'benim', 'beri', 'beş', 'bile', 'bilmeksizin', 'bilerek', 'bin', 'bir', 
    'birçoğu', 'birçok', 'biri', 'birkaçı', 'birkaç', 'birkez', 'birlikte', 'birşey', 
    'birşeyi', 'biz', 'bizden', 'bizi', 'bizim', 'böyle', 'böylece', 'bu', 'buna', 
    'bunda', 'bundan', 'bunlar', 'bunları', 'bunların', 'bunu', 'bunun', 'burada', 
    'büyük', 'cümle', 'çünkü', 'da', 'daha', 'dahi', 'dahil', 'daima', 'dair', 
    'dayanarak', 'de', 'defa', 'değil', 'demek', 'dışında', 'dikkat', 'diğer', 
    'diğeri', 'diğerleri', 'diye', 'dokuz', 'dolayı', 'dolayısıyla', 'dört', 
    'edecek', 'eden', 'ederek', 'edilecek', 'edilen', 'edilmesi', 'ediyor', 
    'eğer', 'elbette', 'en', 'esnasında', 'etmesi', 'etmek', 'etti', 'ettiği', 
    'ettiğini', 'fakat', 'falan', 'filan', 'gene', 'gereği', 'gerek', 'gibi', 
    'göre', 'hakkında', 'halen', 'hangi', 'hangisi', 'hani', 'gibi', 'hem', 'henüz', 
    'hep', 'hepsi', 'her', 'herhangi', 'herkes', 'herkesin', 'hiç', 'hiçbir', 
    'hiçbiri', 'gibi', 'için', 'içinde', 'iki', 'ile', 'ilgili', 'illaki', 'ise', 
    'ışığında', 'itibaren', 'itibariyle', 'kaç', 'kadar', 'karşın', 'katrilyon', 
    'kendi', 'kendilerine', 'kendini', 'kendisi', 'kendisine', 'kendisini', 
    'kez', 'ki', 'kim', 'kime', 'kimi', 'kimin', 'kimisi', 'kimse', 'kırk', 
    'maddeten', 'mı', 'mı', 'milyar', 'milyon', 'mu', 'mü', 'nasıl', 'ne', 
    'neden', 'nedenle', 'nedir', 'nerde', 'nerede', 'nereden', 'nereye', 'nesi', 
    'niye', 'niçin', 'o', 'olan', 'olarak', 'oldu', 'olduğu', 'olduğunu', 
    'olduklarını', 'olmadı', 'olmadığı', 'olmak', 'olması', 'olmayan', 'olmaz', 
    'olsa', 'olsun', 'olup', 'olur', 'olursa', 'oluyor', 'on', 'ona', 'ondan', 
    'onlar', 'onlardan', 'onları', 'onların', 'onu', 'onun', 'orada', 'oysa', 
    'oysaki', 'öbürü', 'önce', 'ötürü', 'ötürü', 'özellikle', 'pek', 'rağmen', 
    'sadece', 'sanki', 'sekiz', 'seksen', 'sen', 'senden', 'seni', 'senin', 
    'siz', 'sizden', 'sizi', 'sizin', 'sonra', 'şayet', 'şey', 'şeyden', 
    'şeyi', 'şeyler', 'şu', 'şuna', 'şundaki', 'şundan', 'şunlar', 'şunları', 
    'şunu', 'şunun', 'tarafından', 'trilyon', 'tüm', 'üç', 'üzere', 'var', 
    'vardı', 've', 'veya', 'veya', 'veya', 'ya', 'yani', 'yapacak', 'yapılan', 
    'yapılması', 'yapmak', 'yaptı', 'yaptığı', 'yaptığını', 'yaptıkları', 
    'yedi', 'yerine', 'yetmiş', 'yine', 'yirmi', 'yoksa', 'yüz', 'zaten'
]


In [8]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = text.split()
    text = [word for word in text if word.lower() not in turkce_stopwords]
    return " ".join(text)

train_dataset["cleaned_mentor_feedback"] = train_dataset["mentor_feedback_text"].apply(clean_text)

## Sentiment Analizi

In [9]:
from transformers import pipeline
sentiment_model = pipeline(
    "sentiment-analysis",
    model="savasy/bert-base-turkish-sentiment-cased",
    tokenizer="savasy/bert-base-turkish-sentiment-cased"
) 

texts = train_dataset["mentor_feedback_text"].fillna("").astype(str).tolist()

results = sentiment_model(
    texts,
    truncation=True,
    max_length=512,
    batch_size=32
)

sentiment_scores = []

for r in results:
    if r["label"].upper() == "POSITIVE":
        sentiment_scores.append(r["score"])
    else:
        sentiment_scores.append(-r["score"])

train_dataset["sentiment_score"] = sentiment_scores

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [10]:
vectorizer = TfidfVectorizer(max_features = 250,ngram_range=(1,2))
feedback_text = vectorizer.fit_transform(train_dataset["cleaned_mentor_feedback"])
tmft = pd.DataFrame(feedback_text.toarray(),columns = vectorizer.get_feature_names_out())#train_mentor_feedback_text

In [11]:
train_dataset["university_tier_num"] = train_dataset["university_tier"].str.extract(r'(\d+)').astype(int)
train_dataset["university_tier_num"] = 5- train_dataset["university_tier_num"]

In [12]:
X_numeric = train_dataset.select_dtypes(include = ["float","int"])
X_numeric = X_numeric.drop(columns = ["application_year","graduation_year"])
X_final = pd.concat([X_numeric,tmft],axis = 1)

## TF-ID ile numerik değerli değişkenlerin Birleşmesi

In [13]:
X = X_final.drop(columns = ["career_success_score"])
y = X_final["career_success_score"]

# XGBOOST Regressor Denemesi

In [14]:
xgb=XGBRegressor( n_estimators=1300,
    learning_rate=0.033,
    max_depth=4,subsample = 0.8,colsample_bytree = 0.9,min_child_weight = 3)

In [15]:
kf = KFold(n_splits = 5,shuffle = True,random_state = 42)
scores = cross_val_score(xgb,X,y,cv = kf,scoring = "r2")
print(f"scores :",scores)
print(f"scores :",scores.mean())

scores : [0.60406534 0.6090721  0.65132952 0.62714304 0.61839561]
scores : 0.6220011225559725


# Test Veri Seti

In [16]:
test_data = pd.read_csv("/kaggle/input/competitions/datathon-2026/test_x.csv")
test_data.columns

Index(['student_id', 'application_year', 'age', 'graduation_year',
       'department', 'university_tier', 'cgpa', 'english_exam_score',
       'attendance_rate', 'failed_courses_count', 'target_role',
       'coding_score', 'problem_solving_score', 'data_structures_score',
       'sql_score', 'machine_learning_score', 'backend_score',
       'frontend_score', 'cloud_score', 'devops_score',
       'project_quality_score', 'real_client_project_count',
       'internship_count', 'internship_duration_months',
       'freelance_project_count', 'hackathon_count', 'hackathon_awards',
       'portfolio_score', 'github_repo_count', 'github_avg_stars',
       'open_source_contribution_count', 'linkedin_profile_score',
       'cv_quality_score', 'technical_interview_score', 'hr_interview_score',
       'communication_score', 'teamwork_score', 'leadership_score',
       'presentation_score', 'certification_count', 'bootcamp_count',
       'applications_sent', 'interviews_attended', 'hobby',
     

In [17]:
test_data["english_exam_score"] = test_data["english_exam_score"].fillna(test_data["english_exam_score"].mean())
test_data["open_source_contribution_count"] = test_data["open_source_contribution_count"].fillna(train_dataset["open_source_contribution_count"].mean())
test_data["internship_duration_months"] = test_data["internship_duration_months"].fillna(test_data["internship_duration_months"].mean())
test_data["hr_interview_score"] = test_data["hr_interview_score"].fillna(test_data["hr_interview_score"].mean())
test_data["linkedin_profile_score"] = test_data["linkedin_profile_score"].fillna(test_data["linkedin_profile_score"].mean())
test_data["portfolio_score"] = test_data["portfolio_score"].fillna(test_data["portfolio_score"].mean())
test_data["github_avg_stars"] = test_data["github_avg_stars"].fillna(test_data["github_avg_stars"].mean())

In [18]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = text.split()
    text = [word for word in text if word.lower() not in turkce_stopwords]
    return " ".join(text)

test_data["cleaned_mentor_feedback"] = test_data["mentor_feedback_text"].apply(clean_text)

### Sentiment

In [19]:
test_texts = train_dataset["mentor_feedback_text"].fillna("").astype(str).tolist()

results = sentiment_model(
    test_texts,
    truncation=True,
    max_length=512,
    batch_size=32
)

sentiment_scores = []

for r in results:
    if r["label"].upper() == "POSITIVE":
        sentiment_scores.append(r["score"])
    else:
        sentiment_scores.append(-r["score"])

test_data["sentiment_score"] = sentiment_scores

## TF-ID

In [20]:
test_feedback_text = vectorizer.transform(test_data["cleaned_mentor_feedback"])
test_tmft = pd.DataFrame(test_feedback_text.toarray(),columns = vectorizer.get_feature_names_out())#train_mentor_feedback_text

In [21]:
test_data["university_tier_num"] = test_data["university_tier"].str.extract(r'(\d+)').astype(int)
test_data["university_tier_num"] = 5- test_data["university_tier_num"]

In [22]:
test_data_numeric = test_data.select_dtypes(include= ["float","int"])

In [23]:
test_data_final = pd.concat([test_data_numeric,test_tmft],axis = 1)
test_data_final = test_data_final.drop(columns = ["application_year","graduation_year"])

In [24]:
train_sutunlari = X.columns.tolist()


for sutun in train_sutunlari:
    if sutun not in test_data_final.columns:
        test_data_final[sutun] = 0


X_gercek_test_ready = test_data_final[train_sutunlari]

# Model Eğitme ve Tahminde Bulunma

In [25]:
xgb.fit(X,y)
y_pred = xgb.predict(X_gercek_test_ready)

In [26]:
submission = pd.DataFrame({
    'student_id': test_data['student_id'],
    'career_success_score': y_pred
})


submission.to_csv("submission_last.csv", index=False)